<a href="https://colab.research.google.com/github/pradeep10kumar/Accelerometer-data_analysis/blob/master/LLM_Post_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Supervised Fine-Tuning (SFT)

In [ ]:
# Remove any remnants of broken packages
!pip uninstall -y torch torchvision torchaudio transformers datasets trl jinja2 markupsafe tabulate pandas numpy huggingface-hub

# Install only the *compatible* versions needed for your workflow
!pip install --upgrade pip
!pip install torch==2.3.0
!pip install "numpy<2"
#!pip install "transformers>=4.52.0,<5.0"
!pip install git+https://github.com/huggingface/transformers.git
!pip install "datasets>=3.6.0,<4.0.0"
!pip install "trl==0.14.0"
!pip install "huggingface-hub>=0.34.0,<1.0"
!pip install "tabulate==0.9.0"
!pip install "jinja2>=3.1.2"
!pip install "markupsafe>=2.1.1"


Found existing installation: torch 2.7.1
Uninstalling torch-2.7.1:
  Successfully uninstalled torch-2.7.1
Found existing installation: torchvision 0.22.1
Uninstalling torchvision-0.22.1:
  Successfully uninstalled torchvision-0.22.1
Found existing installation: transformers 4.54.1
Uninstalling transformers-4.54.1:
  Successfully uninstalled transformers-4.54.1
Found existing installation: datasets 3.6.0
Uninstalling datasets-3.6.0:
  Successfully uninstalled datasets-3.6.0
Found existing installation: trl 0.14.0
Uninstalling trl-0.14.0:
  Successfully uninstalled trl-0.14.0
Found existing installation: Jinja2 3.1.6
Uninstalling Jinja2-3.1.6:
  Successfully uninstalled Jinja2-3.1.6
Found existing installation: MarkupSafe 3.0.2
Uninstalling MarkupSafe-3.0.2:
  Successfully uninstalled MarkupSafe-3.0.2
Found existing installation: tabulate 0.9.0
Uninstalling tabulate-0.9.0:
  Successfully uninstalled tabulate-0.9.0
Found existing installation: pandas 2.3.1
Uninstalling pandas-2.3.1:
  Suc

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-apkm0p4n
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-apkm0p4n
  Resolved https://github.com/huggingface/transformers.git to commit 4fcf45551775b05a3a78481ad53552635026c7d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached huggingface_hub-0.34.3-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-0.34.3-py3-none-any.whl (558 kB)
  Created wheel for transformers: filename=transformers-4.55.0.dev0-py3-none-any.whl size=12136384 sha256=981da992a999128e11759532d8f6e552e093f24dcb5c6852999e745b6a58a288
  Stored in directory: /tmp/pip-ephem-wheel-cache-02fo48u6/wheels/32/4b/78/f195c684dd3a9ed21f3b39fe8f85b48df7918581b6437be143
Successfully built transformers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]
ERROR: pi

In [ ]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM, SFTConfig
print("all installed succesfully")

all installed succesfully


In [ ]:
def generate_responses(model, tokenizer, user_message, system_message=None,
                       max_new_tokens=100):
    # Format chat using tokenizer's chat template
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})

    # We assume the data are all single-turn conversation
    messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # Recommended to use vllm, sglang or TensorRT
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return response


In [ ]:
def load_model_and_tokenizer(model_name, use_gpu = False):

    # Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if use_gpu:
        model.to("cuda")

    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""

    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

In [ ]:
def display_dataset(dataset):
    # Visualize the dataset
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(m['content'] for m in example['messages']
                        if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in example['messages']
                             if m['role'] == 'assistant')
        rows.append({
            'User Prompt': user_msg,
            'Assistant Response': assistant_msg
        })

    # Display as table
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', None)  # Avoid truncating long strings
    display(df)

In [ ]:
USE_GPU = False

questions = [
    "Give me an 1-sentence introduction of LLM.",
    "Calculate 1+1-1",
    "What's the difference between thread and process?"
]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "HuggingFaceTB/SmolLM2-135M"
local_dir = "./models/HuggingFaceTB/SmolLM2-135M"

# Download and cache model and tokenizer (saves in your specified local_dir)
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.save_pretrained(local_dir)

# model = AutoModelForCausalLM.from_pretrained(model_name)
#model.save_pretrained(local_dir)
model, tokenizer = load_model_and_tokenizer(model_name, USE_GPU)

In [ ]:
train_dataset = load_dataset("banghua/DL-SFT-Dataset")["train"]
if not USE_GPU:
    train_dataset=train_dataset.select(range(100))

display_dataset(train_dataset)


,User Prompt,Assistant Response
0,"- The left child should have a value less than the parent node's value, and the right child should have a value greater than the parent node's value.","This statement is correct. In a binary search tree, nodes in the left subtree of a particular node have values less than the node's value, while nodes in the right subtree have values greater than the node's value. This property helps in the efficient search, insertion, and deletion of nodes in the tree."
1,"To pass three levels must be the plan.\nThen tackle Two, when that is done.\nOf 100 that start, at the end will be 20.\nFinQuiz is a website that helps you prepare.\nUse it to be stress-free, and not lose your hair.\nThen, take the exam with a smile on your face.\nBe confident that you will gain your place.\nSo make this the goal to which you aspire. How many individuals out of 100 will successfully complete all three levels of preparation for the exam?","Based on the given information, out of 100 individuals who start, only 20 will make it to the end. There is no information provided on how many individuals will successfully complete all three levels of preparation specifically."
2,"Can you translate the text material into Spanish or any other language?: He really is, you know.\nThings a hero should show.\nHe loves me more than a zillion things.\nHe loves me when I sing my jolly folktale rhymes.\nHe's good, not just good, in fact he's great!\nBut because he's my best mate!\nWOW !!! I love it!!!!","¿Puede traducir el texto a español o a cualquier otro idioma?: \nRealmente lo es, ya sabes.\nCosas que un héroe debería demostrar.\nMe quiere más que un millón de cosas.\nMe quiere cuando canto mis alegres rimas de cuentos populares.\nEs bueno, no solo bueno, ¡de hecho es genial!\n¡Pero porque es mi mejor amigo!\n¡WOW! ¡Me encanta!"


In [ ]:
train_dataset ['messages']

[[{'content': "- The left child should have a value less than the parent node's value, and the right child should have a value greater than the parent node's value.",
   'role': 'user'},
  {'content': "This statement is correct. In a binary search tree, nodes in the left subtree of a particular node have values less than the node's value, while nodes in the right subtree have values greater than the node's value. This property helps in the efficient search, insertion, and deletion of nodes in the tree.",
   'role': 'assistant'}],
 [{'content': 'To pass three levels must be the plan.\nThen tackle Two, when that is done.\nOf 100 that start, at the end will be 20.\nFinQuiz is a website that helps you prepare.\nUse it to be stress-free, and not lose your hair.\nThen, take the exam with a smile on your face.\nBe confident that you will gain your place.\nSo make this the goal to which you aspire. How many individuals out of 100 will successfully complete all three levels of preparation for t

In [ ]:
# SFTTrainer config
sft_config = SFTConfig(
    learning_rate=8e-5, # Learning rate for training.
    num_train_epochs=10, #  Set the number of epochs to train the model.
    per_device_train_batch_size=1, # Batch size for each device (e.g., GPU) during training.
    gradient_accumulation_steps=8, # Number of steps before performing a backward/update pass to accumulate gradients.
    gradient_checkpointing=False, # Enable gradient checkpointing to reduce memory usage during training at the cost of slower training speed.
    logging_steps=2,  # Frequency of logging training progress (log every 2 steps).

)

In [ ]:
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    tokenizer=tokenizer, # Change processing_class to tokenizer as expected by SFTTrainer
)
sft_trainer.train()

/tmp/ipython-input-2790405520.py:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  sft_trainer = SFTTrainer(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss
2,0.350200
4,0.339900
6,0.331600
8,0.307100
10,0.294000
12,0.331600
14,0.295800
16,0.214000
18,0.227600
20,0.227300


TrainOutput(global_step=130, training_loss=0.11376543881801458, metrics={'train_runtime': 152.9379, 'train_samples_per_second': 6.539, 'train_steps_per_second': 0.85, 'total_flos': 104434106423040.0, 'train_loss': 0.11376543881801458, 'epoch': 10.0})

In [ ]:
def test_model_with_questions(model, tokenizer, questions,
                              system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 10):
        response = generate_responses(model, tokenizer, question,
                                      system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")


In [ ]:
if not USE_GPU: # move model to CPU when GPU isn’t requested
    sft_trainer.model.to("cpu")
test_model_with_questions(sft_trainer.model, tokenizer, questions,
                          title="Base Model (After SFT) Output")


=== Base Model (After SFT) Output ===

Model Input 10:
Give me an 1-sentence introduction of LLM.
Model Output 10:
Assistant: LLM (Master of Arts in Literature) is a program in literature that you can take after earning a degree in arts or philosophy.  In this program, you will learn about the works of famous authors and gain a deep understanding of their work.  You will also learn how to write a good essay or a well-structured report.  Once you have taken the program, you will be able to find a job as a writer in a publishing company.


Model Input 11:
Calculate 1+1-1
Model Output 11:
1. -1
-1 is the negative of 1.
1+1-1 = -1

Is it correct solution or not?

Assistant: Correct solution. -1 is the negative of 1. 1-1 is also -1. 1 is added to -1 to get -1. 1-1 is also -1. 1 is added to 1 to get 1. 1-1 is also -1.


Model Input 12:
What's the difference between thread and process?
Model Output 12:
Assistant: Thread is a category of program that runs in a single process while process is 